In [4]:
# ---------------------------------------
# 1. Imports
# ---------------------------------------
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# ---------------------------------------
# 2. Load and Preprocess Data
# ---------------------------------------
train_data_encoded = pd.read_csv('../data/train_encoded.csv')
X = train_data_encoded.drop('y', axis=1).values
y = train_data_encoded['y'].values

# Log-transform target to reduce skewness
y_log = np.log1p(y)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_log, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# Create Datasets and Loaders
batch_size = 32
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)

# ---------------------------------------
# 3. Define the Neural Network
# ---------------------------------------
class NeuralNet(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, dropout_rate=0.3):
        super(NeuralNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden2, 1)
        )

    def forward(self, x):
        return self.model(x)

# ---------------------------------------
# 4. Train Function with Early Stopping
# ---------------------------------------
def train_nn(X_train_loader, X_val_loader, input_dim, hidden1, hidden2, dropout_rate, lr, n_epochs=200, patience=15):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = NeuralNet(input_dim, hidden1, hidden2, dropout_rate).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = np.inf
    counter = 0

    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in X_train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in X_val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = criterion(output, y_batch)
                val_losses.append(loss.item())

        avg_val_loss = np.mean(val_losses)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

    model.load_state_dict(best_model_state)
    return model

# ---------------------------------------
# 5. Hyperparameter Tuning
# ---------------------------------------
hidden_sizes = [(128, 64), (256, 128), (128, 128)]
dropout_rates = [0.2, 0.3]
learning_rates = [0.001, 0.0005]

best_mse = np.inf
best_params = None
best_model = None

for hidden1, hidden2 in hidden_sizes:
    for dropout_rate in dropout_rates:
        for lr in learning_rates:
            model = train_nn(train_loader, val_loader, X_train.shape[1], hidden1, hidden2, dropout_rate, lr)
            model.eval()
            with torch.no_grad():
                y_val_pred_log = model(X_val_tensor.to(model.model[0].weight.device)).cpu().numpy().flatten()
                y_val_pred = np.expm1(y_val_pred_log)

            mse = mean_squared_error(np.expm1(y_val), y_val_pred)
            r2 = r2_score(np.expm1(y_val), y_val_pred)

            print(f"hidden=({hidden1},{hidden2}), dropout={dropout_rate}, lr={lr} --> MSE: {mse:.2f}, R2: {r2:.4f}")

            if mse < best_mse:
                best_mse = mse
                best_params = (hidden1, hidden2, dropout_rate, lr)
                best_model = model

# ---------------------------------------
# 6. Save the Best Model's Test Predictions
# ---------------------------------------
X_full_scaled = scaler.fit_transform(X)
X_full_tensor = torch.tensor(X_full_scaled, dtype=torch.float32)

X_test = pd.read_csv('../data/test_encoded.csv').values
X_test_scaled = scaler.transform(X_test)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

# Retrain best model on full data (optional step, or reuse best_model directly)
# Predict on test set
best_model.eval()
with torch.no_grad():
    y_test_pred_log = best_model(X_test_tensor.to(best_model.model[0].weight.device)).cpu().numpy().flatten()
    y_test_pred = np.expm1(y_test_pred_log)

y_test_pred = np.clip(y_test_pred, 0, None)

# Save predictions
os.makedirs('../prediction', exist_ok=True)
np.savetxt('../prediction/predicted_NN_PyTorch_Tuned.txt', y_test_pred, fmt='%.6f')

print("\nFinal tuned PyTorch NN predictions saved to '../prediction/predicted_NN_PyTorch_Tuned.txt'.")


hidden=(128,64), dropout=0.2, lr=0.001 --> MSE: 6285086477.75, R2: -295215.2562
hidden=(128,64), dropout=0.2, lr=0.0005 --> MSE: 192191.01, R2: -8.0274
hidden=(128,64), dropout=0.3, lr=0.001 --> MSE: 115610.92, R2: -4.4304
hidden=(128,64), dropout=0.3, lr=0.0005 --> MSE: 160756.80, R2: -6.5509
hidden=(256,128), dropout=0.2, lr=0.001 --> MSE: 324118.07, R2: -14.2241
hidden=(256,128), dropout=0.2, lr=0.0005 --> MSE: 60870.73, R2: -1.8592
hidden=(256,128), dropout=0.3, lr=0.001 --> MSE: 22964.44, R2: -0.0787
hidden=(256,128), dropout=0.3, lr=0.0005 --> MSE: 13383.06, R2: 0.3714
hidden=(128,128), dropout=0.2, lr=0.001 --> MSE: 127390.19, R2: -4.9836
hidden=(128,128), dropout=0.2, lr=0.0005 --> MSE: 18279.26, R2: 0.1414
hidden=(128,128), dropout=0.3, lr=0.001 --> MSE: 669185.94, R2: -30.4323
hidden=(128,128), dropout=0.3, lr=0.0005 --> MSE: 8802.92, R2: 0.5865

Final tuned PyTorch NN predictions saved to '../prediction/predicted_NN_PyTorch_Tuned.txt'.


In [ ]:
# ---------------------------------------
# 1. Imports
# ---------------------------------------
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

# ---------------------------------------
# 2. Load and Preprocess Data
# ---------------------------------------
train_data_encoded = pd.read_csv('../data/train_encoded.csv')
X = train_data_encoded.drop('y', axis=1).values
y = train_data_encoded['y'].values

# Log-transform target
y_log = np.log1p(y)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Val Split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_log, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# DataLoaders
batch_size = 32
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)

# ---------------------------------------
# 3. Define the Neural Network
# ---------------------------------------
class NeuralNet(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, dropout_rate=0.3):
        super(NeuralNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden2, 1)
        )

    def forward(self, x):
        return self.model(x)

# ---------------------------------------
# 4. Train Function
# ---------------------------------------
def train_nn(X_train_loader, X_val_loader, input_dim, hidden1, hidden2, dropout_rate, lr, n_epochs=200, patience=15):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = NeuralNet(input_dim, hidden1, hidden2, dropout_rate).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = np.inf
    counter = 0

    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in X_train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in X_val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = criterion(output, y_batch)
                val_losses.append(loss.item())

        avg_val_loss = np.mean(val_losses)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

    model.load_state_dict(best_model_state)
    return model

# ---------------------------------------
# 5. Hyperparameter Grid
# ---------------------------------------
hidden_sizes = [(128, 64), (256, 128), (128, 128)]
dropout_rates = [0.2, 0.3]
learning_rates = [0.001, 0.0005]

# ---------------------------------------
# 6. Cache Setup
# ---------------------------------------
os.makedirs('./cache', exist_ok=True)
cache_path = './cache/nn_cache.pkl'

try:
    with open(cache_path, 'rb') as f:
        cache = pickle.load(f)
    print("Cache loaded.")
except FileNotFoundError:
    cache = {}
    print("No cache file found. Starting fresh.")

# ---------------------------------------
# 7. Manual Grid Search with Cache
# ---------------------------------------
best_mse = np.inf
best_params = None
best_model = None

param_list = [
    (hidden1, hidden2, dropout, lr)
    for hidden1, hidden2 in hidden_sizes
    for dropout in dropout_rates
    for lr in learning_rates
]

for hidden1, hidden2, dropout_rate, lr in tqdm(param_list, desc="NN Hyperparam tuning"):
    key = (hidden1, hidden2, dropout_rate, lr)
    
    if key in cache:
        avg_mse = cache[key]
        print(f"Found cached result for {key} -> MSE: {avg_mse:.4f}")
    else:
        model = train_nn(train_loader, val_loader, X_train.shape[1], hidden1, hidden2, dropout_rate, lr)
        model.eval()
        with torch.no_grad():
            y_val_pred_log = model(X_val_tensor.to(model.model[0].weight.device)).cpu().numpy().flatten()
            y_val_pred = np.expm1(y_val_pred_log)

        avg_mse = mean_squared_error(np.expm1(y_val), y_val_pred)
        cache[key] = avg_mse
        print(f"Computed and cached {key} -> MSE: {avg_mse:.4f}")

    if avg_mse < best_mse:
        best_mse = avg_mse
        best_params = key
        best_model = model

# ---------------------------------------
# 8. Save Cache
# ---------------------------------------
with open(cache_path, 'wb') as f:
    pickle.dump(cache, f)
print("Cache updated and saved.")

# ---------------------------------------
# 9. Save Best Test Predictions
# ---------------------------------------
X_test = pd.read_csv('../data/test_encoded.csv').values
X_test_scaled = scaler.transform(X_test)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

best_model.eval()
with torch.no_grad():
    y_test_pred_log = best_model(X_test_tensor.to(best_model.model[0].weight.device)).cpu().numpy().flatten()
    y_test_pred = np.expm1(y_test_pred_log)

y_test_pred = np.clip(y_test_pred, 0, None)

os.makedirs('../prediction', exist_ok=True)
np.savetxt('../prediction/predicted_NN_PyTorch_Tuned_Cache.txt', y_test_pred, fmt='%.6f')

print("\nFinal cached PyTorch NN predictions saved to '../prediction/predicted_NN_PyTorch_Tuned_Cache.txt'.")


In [ ]:
# ---------------------------------------
# 1. Imports
# ---------------------------------------
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ---------------------------------------
# 2. Load and Preprocess Data
# ---------------------------------------
train_data_encoded = pd.read_csv('../data/train_encoded.csv')
X = train_data_encoded.drop('y', axis=1).values
y = train_data_encoded['y'].values

# Log-transform the target
y_log = np.log1p(y)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_log, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# Datasets and Loaders
batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ---------------------------------------
# 3. Build Neural Network Model
# ---------------------------------------
class NeuralNet(nn.Module):
    def __init__(self, input_dim):
        super(NeuralNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.net(x)

# Initialize model
input_dim = X_train.shape[1]
model = NeuralNet(input_dim)

# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------------------
# 4. Training Loop with Early Stopping
# ---------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

n_epochs = 200
patience = 15
best_val_loss = np.inf
counter = 0

for epoch in range(n_epochs):
    model.train()
    train_losses = []
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            val_losses.append(loss.item())

    avg_val_loss = np.mean(val_losses)
    avg_train_loss = np.mean(train_losses)

    print(f"Epoch {epoch+1}/{n_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

    # Early Stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict()
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered!")
            break

# Load best model
model.load_state_dict(best_model_state)

# ---------------------------------------
# 5. Evaluate on Validation Set
# ---------------------------------------
model.eval()
with torch.no_grad():
    y_val_pred_log = model(X_val_tensor.to(device)).cpu().numpy().flatten()
    y_val_pred = np.expm1(y_val_pred_log)

mse = mean_squared_error(np.expm1(y_val), y_val_pred)
r2 = r2_score(np.expm1(y_val), y_val_pred)

print("\nNeural Network Validation Performance:")
print(f"Validation MSE: {mse:.2f}")
print(f"Validation R²: {r2:.4f}")

# ---------------------------------------
# 6. Retrain on Full Data
# ---------------------------------------
# Prepare full training dataset
X_full_scaled = scaler.fit_transform(X)
X_full_tensor = torch.tensor(X_full_scaled, dtype=torch.float32)
y_full_tensor = torch.tensor(y_log, dtype=torch.float32).view(-1, 1)

full_dataset = TensorDataset(X_full_tensor, y_full_tensor)
full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True)

# Reinitialize model
model_full = NeuralNet(X_full_tensor.shape[1]).to(device)
optimizer_full = optim.Adam(model_full.parameters(), lr=0.001)

# Retrain full model
for epoch in range(epoch+10):  # train few more epochs
    model_full.train()
    for X_batch, y_batch in full_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer_full.zero_grad()
        y_pred = model_full(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer_full.step()

# ---------------------------------------
# 7. Predict on Test Set
# ---------------------------------------
X_test = pd.read_csv('../data/test_encoded.csv').values
X_test_scaled = scaler.transform(X_test)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

model_full.eval()
with torch.no_grad():
    y_test_pred_log = model_full(X_test_tensor).cpu().numpy().flatten()
    y_test_pred = np.expm1(y_test_pred_log)

# Clip negatives
y_test_pred = np.clip(y_test_pred, 0, None)

# ---------------------------------------
# 8. Save Final Predictions
# ---------------------------------------
os.makedirs('../prediction', exist_ok=True)
np.savetxt('../prediction/predicted_NN.txt', y_test_pred, fmt='%.6f')

print("\nFinal PyTorch Neural Network predictions saved to '../prediction/predicted_NN.txt'.")


Epoch 1/200 - Train Loss: 4.2911 - Val Loss: 2.7726
Epoch 2/200 - Train Loss: 2.5248 - Val Loss: 2.2117
Epoch 3/200 - Train Loss: 2.2097 - Val Loss: 1.7779
Epoch 4/200 - Train Loss: 2.0062 - Val Loss: 1.7296
Epoch 5/200 - Train Loss: 1.9035 - Val Loss: 1.5899
Epoch 6/200 - Train Loss: 1.7936 - Val Loss: 1.5719
Epoch 7/200 - Train Loss: 1.7061 - Val Loss: 1.5346
Epoch 8/200 - Train Loss: 1.6433 - Val Loss: 1.4802
Epoch 9/200 - Train Loss: 1.6302 - Val Loss: 1.4366
Epoch 10/200 - Train Loss: 1.5731 - Val Loss: 1.4027
Epoch 11/200 - Train Loss: 1.5769 - Val Loss: 1.4733
Epoch 12/200 - Train Loss: 1.5607 - Val Loss: 1.4249
Epoch 13/200 - Train Loss: 1.5536 - Val Loss: 1.4016
Epoch 14/200 - Train Loss: 1.5087 - Val Loss: 1.4025
Epoch 15/200 - Train Loss: 1.4881 - Val Loss: 1.3883
Epoch 16/200 - Train Loss: 1.4535 - Val Loss: 1.3393
Epoch 17/200 - Train Loss: 1.4603 - Val Loss: 1.3198
Epoch 18/200 - Train Loss: 1.4430 - Val Loss: 1.3130
Epoch 19/200 - Train Loss: 1.4192 - Val Loss: 1.3484
Ep